# Kámárí · Data Pipeline  (single notebook · Google Colab)

One standard, end-to-end pipeline:

**1. Gather** licence-cleared datasets → **2. Preprocess** (face-align, crop 224, quality, ITA/skin band, manifest) → **3. EDA** (distributions + fairness + report) → **4. Publish to Hugging Face**.

After this runs, the data lives on HF and the **Modal training scripts pull it directly**:

| HF repo | Visibility | Holds | Used by |
|---|---|---|---|
| `<ns>/kamari-faces-v0` | **private** | aligned crops + train/benchmark manifests | Modal CNN training |
| `<ns>/dataset-registry-v0` | private | manifests + registry + licences + EDA + card | provenance |
| `<ns>/kamari-safe-open-v0` | private | frozen benchmark manifest | benchmarking |

**Colab setup:** add Secrets `KAMARI_REPO_URL`, `HF_TOKEN`, `HF_NAMESPACE`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (+ optional `FAGE_HF_REPO`). Use a **GPU runtime** for preprocessing. Crops are pushed to a **private** repo only — raw faces are never made public.

In [ ]:
# --- Kámárí bootstrap: works on Google Colab and locally ---
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import userdata
        for _k in ['HF_TOKEN','HF_NAMESPACE','KAGGLE_USERNAME','KAGGLE_KEY','FAGE_HF_REPO','KAMARI_REPO_URL']:
            try: os.environ.setdefault(_k, userdata.get(_k))
            except Exception: pass
    except Exception: pass
    if not pathlib.Path('/content/kamari/data').exists():
        os.system(f"git clone {os.environ.get('KAMARI_REPO_URL','')} /content/kamari")
    os.system('pip install -q -r /content/kamari/requirements-data.txt')
    REPO = pathlib.Path('/content/kamari')
else:
    REPO = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (c/'data'/'registry').exists()), pathlib.Path.cwd().parent)
    from dotenv import load_dotenv; load_dotenv(REPO/'.env')
sys.path.append(str(REPO))
print('REPO =', REPO, '| Colab:', IN_COLAB)

In [ ]:
import yaml, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from data.adapters import REGISTERED
from data import face_utils as F
from data.manifest_schema import empty_manifest, MANIFEST_COLUMNS, ita_to_band, validate
from data.hf_utils import HF, dataset_card

assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN'
RAW = REPO/'data'/'raw'; CROPS = REPO/'data'/'processed'/'crops_224'
MAN = REPO/'data'/'manifests'; CARDS = REPO/'data'/'cards'; FIG = CARDS/'eda'
for p in (RAW, CROPS, MAN, FIG): p.mkdir(parents=True, exist_ok=True)
have_kaggle = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
QUALITY_MIN = 0.30
registry = yaml.safe_load(open(REPO/'data'/'registry'/'datasets_free_open.yaml'))
hf = HF()
print('namespace:', hf.namespace, '| kaggle:', have_kaggle, '| adapters:', list(REGISTERED))

## 1 · Gather
Download only what the licence allows. Agreement-only datasets are dropped into `data/raw/<name>/` manually.

In [ ]:
import zipfile, urllib.request

def dl_url(url, dest_zip, out):
    out = pathlib.Path(out)
    if out.exists() and any(out.iterdir()): print('skip', out.name); return
    out.mkdir(parents=True, exist_ok=True); urllib.request.urlretrieve(url, dest_zip)
    with zipfile.ZipFile(dest_zip) as z: z.extractall(out); print('extracted', out.name)

def dl_kaggle(slug, out):
    if not have_kaggle: print('no kaggle, skip', slug); return
    pathlib.Path(out).mkdir(parents=True, exist_ok=True)
    os.system(f'kaggle datasets download -d {slug} -p "{out}" --unzip')

dl_kaggle('jangedoo/utkface-new', RAW/'utkface')
try:
    dl_url('https://drive.google.com/uc?export=download&id=1Z1RqRo0_JiavaZw2yzZG6WETdZQ8qX86', RAW/'fairface.zip', RAW/'fairface')
except Exception as e: print('FairFace: fetch manually ->', e)
if os.environ.get('FAGE_HF_REPO'):
    from huggingface_hub import snapshot_download
    snapshot_download(os.environ['FAGE_HF_REPO'], repo_type='dataset', local_dir=str(RAW/'fage_v2'), token=os.environ['HF_TOKEN'])
for sub in sorted(RAW.glob('*')):
    if sub.is_dir(): print(f'{sub.name:18s} {sum(1 for _ in sub.rglob("*") if _.is_file()):>8d} files')

## 2 · Preprocess
Every image, identically: adapter → detect → eye-align → crop 224 → quality → ITA/skin band → manifest row.

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
rows = []
for name, mod in REGISTERED.items():
    root = RAW/name.lower()
    if not root.exists(): print('skip (not downloaded):', name); continue
    n = sum(1 for r in mod.iter_rows(str(root)) for _ in [rows.append(r)])
    print(f'{name}: {n} raw rows')
print('total raw rows:', len(rows))

def process(row):
    fp = pathlib.Path(row['path'])
    if not fp.exists(): return None
    try: rgb = np.array(Image.open(fp).convert('RGB'))
    except Exception: return None
    boxes, probs, lms = F.detect_faces(rgb, device=DEVICE)
    if boxes is None or len(boxes) == 0: return None
    i = int(np.argmax(probs)); crop = F.align_and_crop(rgb, lms[i], size=224)
    ita = F.estimate_ita(crop); b = boxes[i]
    row.update(face_quality=F.quality_score(crop), skin_ita=ita, skin_band=ita_to_band(ita),
               bbox=f"{int(b[0])},{int(b[1])},{int(b[2]-b[0])},{int(b[3]-b[1])}")
    out = CROPS/f"{row['image_id'].replace('/', '__')}.jpg"
    Image.fromarray(crop).save(out, quality=95)
    row['path'] = str(out.relative_to(REPO))
    return row

kept = [pr for r in tqdm(rows) if (pr := process(r)) and (pr.get('face_quality') or 0) >= QUALITY_MIN]
print(f'kept {len(kept)} / {len(rows)}')

In [ ]:
# Build + validate manifests, derive splits, save locally
df = pd.concat([empty_manifest(), pd.DataFrame(kept)], ignore_index=True)[MANIFEST_COLUMNS]
df.loc[df['eval_ok'] == True, 'split'] = 'benchmark'
print('validation:', validate(df) or 'OK')
public = df.copy()
train = df[(df['train_ok'] == True) & (df['split'] != 'benchmark')].copy()
bench = df[df['split'] == 'benchmark'].copy()
for d, n in [(public,'public'), (train,'train'), (bench,'benchmark')]:
    d.to_parquet(MAN/f'manifest_{n}_v0.parquet', index=False)
print('rows:', {'public':len(public),'train':len(train),'benchmark':len(bench)})

## 3 · EDA
Safety + fairness first: the 13–21 boundary and dark-skin coverage.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
band = public[(public['age'] >= 13) & (public['age'] <= 21)]
dark = public[public['skin_band'].isin(['brown','dark'])]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
public['age'].dropna().hist(bins=40, ax=ax[0]); ax[0].set_title('Age')
public['skin_band'].value_counts().plot.bar(ax=ax[1], title='Skin band')
public['dataset'].value_counts().plot.bar(ax=ax[2], title='Dataset')
plt.tight_layout(); plt.savefig(FIG/'overview.png', dpi=120); plt.show()
print(f'boundary 13–21: {len(band)} | dark/brown: {len(dark)} ({100*len(dark)/max(len(public),1):.1f}%)')

lines = ['# Kámárí Data Quality Report (v0)\n', f'- Total: {len(public)}',
  f'- Exact-age: {int((public.age_exact==True).sum())}', f'- Minors: {int((public.is_minor==True).sum())}',
  f'- Boundary 13–21: {len(band)}', f'- Dark/brown skin: {len(dark)}',
  '\n## Datasets\n', public["dataset"].value_counts().to_markdown(),
  '\n## Skin band\n', public["skin_band"].value_counts().to_markdown()]
(CARDS/'data_quality_report.md').write_text('\n'.join(map(str, lines)))
print('wrote data_quality_report.md')

## 4 · Publish to Hugging Face
The one upload step. Crops + manifests → **private** `kamari-faces-v0` (Modal trains from here); provenance → `dataset-registry-v0`; frozen split → `kamari-safe-open-v0`.

In [ ]:
# 4a · kamari-faces-v0 (PRIVATE): crops + manifests with paths rewritten to crops/<file>
import shutil, tempfile
FACES = 'kamari-faces-v0'
hf.ensure_repo(FACES, 'dataset', private=True)
tmp = pathlib.Path(tempfile.mkdtemp())/'crops'; tmp.mkdir(parents=True)
for p in public['path']:
    src = REPO/p
    if src.exists(): shutil.copy(src, tmp/src.name)
hf.upload_folder(str(tmp), FACES, repo_type='dataset', path_in_repo='crops')
for d, n in [(public,'public'), (train,'train'), (bench,'benchmark')]:
    d2 = d.copy(); d2['path'] = d2['path'].map(lambda p: f"crops/{pathlib.Path(p).name}")
    hf.push_manifest(d2, FACES, split=n)
print('faces ->', f'https://huggingface.co/datasets/{hf.repo_id(FACES)} ({len(list(tmp.glob("*")))} crops)')

In [ ]:
# 4b · provenance (dataset-registry-v0) + frozen benchmark (kamari-safe-open-v0)
DS, BENCH = 'dataset-registry-v0', 'kamari-safe-open-v0'
for n, d in [('public',public),('train',train),('benchmark',bench)]:
    hf.push_manifest(d, DS, split=n)
for local, rp in [(REPO/'data'/'registry'/'datasets_free_open.yaml','registry/datasets_free_open.yaml'),
                  (REPO/'data'/'registry'/'licenses.md','registry/licenses.md'),
                  (CARDS/'data_quality_report.md','reports/data_quality_report.md')]:
    if local.exists(): hf.upload_file(str(local), DS, rp, repo_type='dataset')
for png in FIG.glob('*.png'): hf.upload_file(str(png), DS, f'reports/eda/{png.name}', repo_type='dataset')
card = dataset_card(hf.namespace, len(public), sorted(public['dataset'].unique().tolist()))
(CARDS/'DATASET_CARD.md').write_text(card)
hf.upload_file(str(CARDS/'DATASET_CARD.md'), DS, 'README.md', repo_type='dataset')
hf.push_manifest(bench, BENCH, split='benchmark')
hf.upload_file(str(CARDS/'DATASET_CARD.md'), BENCH, 'README.md', repo_type='dataset')

print('Done. Hugging Face artifacts:')
for r in [FACES, DS, BENCH]: print('  ', f'https://huggingface.co/datasets/{hf.repo_id(r)}')
print('\nModal CNN training reads from', hf.repo_id(FACES))